# 尾盘有符号杠杆残差对Stage1联合现金/上限局部净价值的信息门

固定2018–2020原架构OOF训练、2021–2022完整评价；原39项加自身状态及早盘收益为对照，只新增前一交易日尾盘残差。源PIT未认证，信息门不等于完整f696资金账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json,shlex,tempfile
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_ssh
root=Path.cwd();card=root/'research/stage1_late_residual_joint_value_information_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
source_card=root/s['coordinate_card'];minute_card=root/s['minute_card'];label_card=root/s['local_label_card'];ref_card=root/s['reference_card']
assert sha(source_card/'plan.json')==s['coordinate_plan_sha256'] and sha(source_card/'result.json')==s['coordinate_result_sha256']
source_result=json.loads((source_card/'result.json').read_text());assert source_result['source_novelty_pass'] and source_result['derived_coordinate_csv_sha256']==s['coordinate_csv_sha256']
for c,key in [(minute_card,'minute_summary_sha256'),(label_card,'local_label_summary_sha256'),(ref_card,'reference_summary_sha256')]:assert sha(c/'summary.json')==s[key]
for p,key in [(root/s['core_provider']/'qlib_store_manifest.json','core_manifest_sha256'),(root/s['daily_provider']/'qlib_store_manifest.json','daily_manifest_sha256'),(root/'src/analysis/seed_disagreement_information.py','bootstrap_module_sha256')]:assert sha(p)==s[key]
sm=json.loads((minute_card/'summary.json').read_text());sl=json.loads((label_card/'summary.json').read_text());sr=json.loads((ref_card/'summary.json').read_text())
assert sm['nas_evidence']['sha256']==s['minute_NAS_sha256'] and sl['nas_readback']['verified'] and sl['nas_readback']['sha256']==s['local_label_NAS_sha256']
assert sr['nas_readback']['verified'] and sr['nas_readback']['sha256']==s['reference_NAS_sha256']
for c,exp,rid,art,key in [(minute_card,'minute_experiment_id','minute_recorder_id','inputs/minute_features','minute_features_sha256'),(label_card,'local_label_experiment_id','local_label_recorder_id','analysis/labels','local_label_artifact_sha256'),(ref_card,'reference_experiment_id','reference_recorder_id','inputs/source_frame','reference_frame_sha256'),(ref_card,'reference_experiment_id','reference_recorder_id','paths/reference_0','reference_path_sha256')]:
 assert sha(c/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
qlib.init(provider_uri=str((root/s['core_provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((minute_card/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['minute_experiment_id'],recorder_id=s['minute_recorder_id']);old=rec.load_object('inputs/minute_features')
assert len(old)==2008
bars=D.features(['SOXL','SOXX'],['$open','$close','$price_valid'],start_time='2018-10-15',end_time='2026-10-09',freq='1min').reset_index()
bars['datetime']=pd.to_datetime(bars.datetime);bars['date']=bars.datetime.dt.normalize()
avail=pd.to_datetime(old.minute_feature_available_at,utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None)
close_by_day=avail-pd.Timedelta(minutes=1);close_by_day.index=pd.DatetimeIndex(old.index).normalize()
records=[]
for (day,symbol),g in bars.groupby(['date','instrument'],sort=True):
 day=pd.Timestamp(day).normalize();symbol=str(symbol).upper()
 if day not in close_by_day.index:continue
 closing=pd.Timestamp(close_by_day.loc[day])
 if pd.isna(closing):records.append((day,symbol,None));continue
 late=g[(g.datetime>closing-pd.Timedelta(minutes=30))&(g.datetime<=closing)].sort_values('datetime')
 expected=pd.date_range(closing-pd.Timedelta(minutes=29),closing,freq='min')
 valid=bool(len(late)==30 and pd.DatetimeIndex(late.datetime).equals(expected))
 if valid:
  val=late[['$open','$close','$price_valid']].to_numpy(float)
  valid=bool(np.isfinite(val).all() and (val[:,:2]>0).all() and (val[:,2]>0).all())
 ret=float(np.log(late['$close'].to_numpy(float)/late['$open'].to_numpy(float)).sum()) if valid else None
 records.append((day,symbol,ret))
source=pd.DataFrame(records,columns=['date','symbol','late_return30']);assert source.groupby('date').symbol.nunique().le(2).all()
wide=source.pivot(index='date',columns='symbol',values='late_return30').reindex(old.index)
known=wide.SOXL.notna()&wide.SOXX.notna()&old.minute_core_ready.astype(bool)
coord=(wide.SOXL-3*wide.SOXX).where(known)
values=coord.dropna().sort_index();digest=hashlib.sha256(values.to_csv(index=True,float_format='%.17g').encode()).hexdigest()
assert digest==s['coordinate_csv_sha256'] and int(coord.notna().sum())==2005
qlib.init(provider_uri=str((root/s['daily_provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day'));assert len(calendar)==1061
with R.uri_context(str((ref_card/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['reference_experiment_id'],recorder_id=s['reference_recorder_id']);frame=rec.load_object('inputs/source_frame');bundle=rec.load_object('paths/reference_0')
with R.uri_context(str((label_card/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['local_label_experiment_id'],recorder_id=s['local_label_recorder_id']);labels=rec.load_object('analysis/labels')
def indexed(value):
 copy=value.copy();copy['trade_date']=pd.to_datetime(copy.trade_date).dt.normalize()
 assert not copy.trade_date.duplicated().any()
 return copy.set_index('trade_date').reindex(calendar)
frame=indexed(frame);replay=indexed(bundle['replay']);targets=indexed(bundle['targets']);labels=indexed(labels)
assert len(frame)==len(replay)==len(targets)==len(labels)==1061
base=list(sm['input_audit']['original_features']);assert len(base)==39 and set(base).issubset(frame.columns)
signal=pd.DatetimeIndex(frame.signal_date);assert signal.equals(pd.DatetimeIndex(labels.signal_date))
coordinate=coord.reindex(signal);coordinate.index=calendar;coordinate.name='prior_late_signed_leverage_residual30'
available=pd.to_datetime(old.minute_feature_available_at,utc=True).reindex(signal);available.index=calendar
receiver_clock=pd.to_datetime(labels.decision_at,utc=True)
assert (available[coordinate.notna()]<receiver_clock[coordinate.notna()]).all()
state_names=['portfolio_drawdown','current_beta_cap','risk_state','raw_beta','projected_cash','delta_target_beta','early_ret_soxx','early_ret_soxl']
state=pd.DataFrame({'portfolio_drawdown':replay.portfolio_drawdown,'current_beta_cap':replay.current_beta_cap,'risk_state':replay.risk_state.astype(float),
                    'raw_beta':3*frame.soxl_weight+frame.soxx_weight,'projected_cash':targets.cash_weight,'delta_target_beta':labels.delta_target_beta,
                    'early_ret_soxx':frame.early_ret_soxx,'early_ret_soxl':frame.early_ret_soxl},index=calendar).astype(float)
assert not np.isinf(state.to_numpy()).any()
baseline_names=[*base,*state_names];all_features=frame[base].join(state).join(coordinate)
train_dates=calendar[calendar<pd.Timestamp('2021-01-01')]
corr=all_features.loc[train_dates,baseline_names+[coordinate.name]].corr(method='spearman')[coordinate.name]
max_corr=float(corr.loc[baseline_names].abs().max())
y=labels.joint_local_fee_net_advantage.astype(float)
source_known=coordinate.notna();eligible=labels.eligible.astype(bool)&y.notna()
label_available=pd.to_datetime(labels.label_available_at,utc=True).dt.tz_convert('America/New_York').dt.tz_localize(None)
train=eligible&source_known&(calendar>=pd.Timestamp('2018-10-15'))&(calendar<=pd.Timestamp('2020-12-31'))&(label_available<pd.Timestamp('2021-01-01'))
evaluation=eligible&source_known&(calendar>=pd.Timestamp('2021-01-01'))&(calendar<=pd.Timestamp('2022-12-30'))&(label_available<pd.Timestamp('2023-01-01'))
preflight={'status':'source_clock_novelty_and_support_pass' if max_corr<.8 and int(train.sum())>=357 and int(evaluation.sum())==329 else 'source_or_support_gate_failed_before_fit',
           'plan_sha256':sha(plan_path),'calendar_rows':len(calendar),'source_known_on_calendar':int(source_known.sum()),'train_mature_known':int(train.sum()),'eval_mature_known':int(evaluation.sum()),
           'eval_2021':int(evaluation[calendar.year==2021].sum()),'eval_2022':int(evaluation[calendar.year==2022].sum()),'max_abs_train_spearman_vs_baseline':max_corr,
           'source_Recorder_object_reads':4,'Qlib_1min_feature_reads':1,'Qlib_day_calendar_reads':1,'new_model_fits':0}
(card/'preflight_result.json').write_text(json.dumps(preflight,ensure_ascii=False,indent=2)+'\n')
assert preflight['status']=='source_clock_novelty_and_support_pass' and preflight['eval_2021']>=70 and preflight['eval_2022']>=240
static=pd.concat({'feature':all_features,'label':pd.DataFrame({'joint_local_fee_net_advantage':y}),
                  'meta':pd.DataFrame({'train_mature':train,'eval_mature':evaluation})},axis=1)
static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2018-10-15','2020-12-31'),'eval':('2021-01-01','2022-12-30')})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('eval',col_set=['feature','label','meta'])
tr=tr.loc[tr[('meta','train_mature')].astype(bool)];he=he.loc[he[('meta','eval_mature')].astype(bool)]
assert len(tr)==int(train.sum()) and len(he)==329
xtr=tr['feature'];xhe=he['feature'];ytr=tr[('label','joint_local_fee_net_advantage')].to_numpy(float);yhe=he[('label','joint_local_fee_net_advantage')].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
run_state={'status':'fit_attempted','plan_sha256':sha(plan_path),'new_fit_attempts':0,'new_fits_completed':0,'bootstrap_attempts':0}
def save_state(): (card/'run_state.json').write_text(json.dumps(run_state,indent=2)+'\n')
save_state()
def model():return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
baseline=model();candidate=model();candidate_names=baseline_names+[coordinate.name]
run_state['new_fit_attempts']=1;save_state();baseline.fit(xtr[baseline_names],ytr);run_state['new_fits_completed']=1;save_state()
run_state['new_fit_attempts']=2;save_state();candidate.fit(xtr[candidate_names],ytr);run_state['new_fits_completed']=2;save_state()
p0=baseline.predict(xhe[baseline_names]);p1=candidate.predict(xhe[candidate_names]);d=(yhe-p0)**2-(yhe-p1)**2
mean_d=(yhe-np.mean(ytr))**2-(yhe-p1)**2
score=pd.DataFrame({'trade_date':pd.DatetimeIndex(he.index.get_level_values('datetime')),'target':yhe,'baseline':p0,'candidate':p1,'paired_improvement':d,'improvement_vs_train_mean':mean_d})
by_year={str(y):float(score.loc[score.trade_date.dt.year.eq(y),'paired_improvement'].mean()) for y in (2021,2022)}
rank={str(y):float(score.loc[score.trade_date.dt.year.eq(y),['candidate','target']].corr(method='spearman').iloc[0,1]) for y in (2021,2022)}
metrics={'added_standardized_coefficient':float(candidate.named_steps['ridge'].coef_[-1]),'MSE_baseline':float(np.mean((yhe-p0)**2)),
         'MSE_candidate':float(np.mean((yhe-p1)**2)),'MSE_train_mean':float(np.mean((yhe-np.mean(ytr))**2)),
         'paired_improvement':float(np.mean(d)),'improvement_vs_train_mean':float(np.mean(mean_d)),
         'annual_paired_improvement':by_year,'annual_candidate_rank_spearman':rank}
full_calendar=calendar[calendar>=pd.Timestamp('2021-01-01')];assert len(full_calendar)==503
full=pd.Series(np.nan,index=full_calendar,dtype=float);full.loc[score.trade_date.to_numpy()]=d
assert int(full.notna().sum())==len(score)
fixed=plan['information_gate']['paired_full_503_day_calendar_stationary_block_lower_positive']
bootstrap,_,means=full_calendar_bootstrap(full.to_numpy(float),mean_block=fixed['mean_block'],resamples=fixed['resamples'],seed=fixed['seed'],alpha=fixed['one_sided_alpha'])
run_state['bootstrap_attempts']=1;save_state()
gates={'negative_coefficient':metrics['added_standardized_coefficient']<0,
       'beats_baseline':metrics['paired_improvement']>0,'beats_train_mean':metrics['improvement_vs_train_mean']>0,
       'both_eval_years':all(v>0 for v in by_year.values()),'both_year_ranks':all(np.isfinite(v) and v>0 for v in rank.values()),
       'positive_block_lower':bootstrap['lower_bound']>0}
decision='conditional_late_residual_joint_value_information_pass_source_PIT_needed_before_f696_account' if all(gates.values()) else 'not_promoted_fixed_late_residual_joint_value_information_failed'
result={'decision':decision,'plan_sha256':sha(plan_path),'preflight':preflight,'train_mature_known':len(tr),'eval_mature_known':len(he),
        'metrics':metrics,'bootstrap':bootstrap,'gates':gates,'information_pass':all(gates.values()),
        'new_market_GETs':0,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'new_Recorders':1,
        'historical_first_receipt_qualified':False,'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Historical one-minute first-live receipt and final revisions remain unqualified','2018-2022 original-architecture OOF local fee-net labels are not the exact f696 full account','A local information pass cannot prove SOXX wealth, risk, costs or independent forward']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=card.name,uri=str((card/'qlib_recorder').resolve())):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset,coordinate_source_sha256=digest)
 rec.save_objects(artifact_path='models',baseline=baseline,candidate=candidate)
 rec.save_objects(artifact_path='predictions',held_scored=score)
 rec.save_objects(artifact_path='analysis',result=result,full_eval_calendar=full,bootstrap_replicate_means=means)
 rec.log_metrics(eval_paired_MSE_improvement=metrics['paired_improvement'],block_lower=bootstrap['lower_bound'],new_model_fits=2,new_native_accounts=0)
 rid,eid=rec.id,str(rec.experiment_id)
summary={'experiment':card.name,'decision':decision,'objective_achieved':False,'plan_sha256':sha(plan_path),'result':'result.json',
         'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'train_mature_known':len(tr),'eval_mature_known':len(he),
         'metrics':metrics,'bootstrap':bootstrap,'gates':gates,'new_model_fits':2,'new_native_accounts':0,
         'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/dataset','models/*','predictions/held_scored','analysis/bootstrap_replicate_means']}}
binding=json.loads((label_card/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=card.name,source_plan='plan.json',as_of='2022-12-30')
(card/'nas_binding.json').write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
with tempfile.TemporaryDirectory(prefix='late-residual-joint-readback-') as temp:
 restored=Path(temp)/'card';restored.mkdir();materialize_nas_evidence(card/'nas_binding.json',card=restored)
 for art in ('models/baseline','models/candidate','predictions/held_scored'):
  rel=Path('qlib_recorder')/eid/rid/'artifacts'/art;assert sha(card/rel)==sha(restored/rel)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
run_state.update(status='completed',recorder_id=rid,NAS_archive_sha256=remote);save_state()
print({'decision':decision,'preflight':preflight,'metrics':metrics,'bootstrap':bootstrap,'gates':gates,'recorder_id':rid,'NAS_sha256':remote})


{'decision': 'not_promoted_fixed_late_residual_joint_value_information_failed', 'preflight': {'status': 'source_clock_novelty_and_support_pass', 'plan_sha256': '65caaefdd49c9baff2210743412a9d41f6b519545907dd402333794cfbb96df0', 'calendar_rows': 1061, 'source_known_on_calendar': 1059, 'train_mature_known': 357, 'eval_mature_known': 329, 'eval_2021': 82, 'eval_2022': 247, 'max_abs_train_spearman_vs_baseline': 0.1339021751689822, 'source_Recorder_object_reads': 4, 'Qlib_1min_feature_reads': 1, 'Qlib_day_calendar_reads': 1, 'new_model_fits': 0}, 'metrics': {'added_standardized_coefficient': -0.0001243084164116766, 'MSE_baseline': 9.485402466105757e-06, 'MSE_candidate': 9.316190450969133e-06, 'MSE_train_mean': 5.665950684354107e-06, 'paired_improvement': 1.6921201513662067e-07, 'improvement_vs_train_mean': -3.6502397666150288e-06, 'annual_paired_improvement': {'2021': 4.960848909941692e-08, '2022': 2.0891844888176527e-07}, 'annual_candidate_rank_spearman': {'2021': -0.021735817873625562, '2

## 中文结论

# 前日尾盘杠杆残差：联合现金/风险上限含费局部信息门失败

在原架构2018–2022 OOF 0bp Tiered 训练账本上，只检验前一交易日最终30根有效一分钟bar的**有符号SOXL−3×SOXX残差**能否增加次日09:46现金转SOXX、必要时释放0.10 beta的单步含费局部净价值信息。原39项、当时账户回撤/beta上限/风险状态、原目标与同日09:45 SOXX/SOXL早盘收益为匹配对照；两条Ridge(alpha=1)各一次拟合，没有阈值、模型搜索或完整资金账户。冻结来源坐标哈希再生成一致；1061日完整Qlib日历保留，来源已知且标签成熟的训练 **357**、评价 **329**（2021为82、2022为247）。

新增标准化系数 **−0.00012431**，符合事前负号。评价MSE候选 **9.31619×10⁻⁶**、匹配对照 **9.48540×10⁻⁶**、训练标签均值 **5.66595×10⁻⁶**；相对对照改善 **+1.69212×10⁻⁷**，相对训练均值却恶化 **−3.65024×10⁻⁶**。2021/2022配对改善 **+4.96085×10⁻⁸/+2.08918×10⁻⁷**，但候选排序Spearman **−0.0217/+0.1068**。完整503日历、平均块长63日、10000次预定区块检验下界 **−1.49703×10⁻⁸**。六项固定门 **3/6**，训练均值、2021排序与区块下界三门失败。

决定 `not_promoted_fixed_late_residual_joint_value_information_failed`。不因相对较弱旧对照的MSE改善就降低训练均值门、只取2022、翻转残差或调整尾盘窗口；不运行精确f696资金账户。该一分钟来源的历史首次直播接收/最终版本未认证，原架构局部共同复位标签也不是2023+完整自身状态账户。2次拟合、1次区块检验、0新行情GET、0经济账户；Qlib Recorder `bd02a4dcd99c4e5caf14d70175225b5e` 的模型/预测已从NAS压缩包定向恢复，归档SHA-256 `d35d81e2a8071a0b61a0793d00386290bcbdc5e78e4f46ce9af0c5dd4045de99`。SOXX v2目标与独立前向均未达成。
